In [1]:
import pandas as pd
from docling.document_converter import DocumentConverter
import pandas as pd
import re
import json
import os

c:\Users\juan.s\Documents\Cerdificados_de_origem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_and_build_cache(pdf_path, cache_json_path):
    """Procesa el PDF con Docling UNA SOLA VEZ y guarda el resultado en disco."""
    print("⚡ Procesando PDF con Docling (esto solo ocurrirá la primera vez)...")
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    doc = result.document

    all_rows = []

    for num, table in enumerate(doc.tables):
        df = table.export_to_dataframe()
        
        col_ncm = None
        col_mercosul = None

        for col in df.columns:
            col_str = str(col).strip()
            if 'NCM' in col_str:
                col_ncm = col
            elif 'MERCOSUL' in col_str.upper():
                col_mercosul = col

        if col_ncm and col_mercosul:
            for _, row in df.iterrows():
                ncm_val = row[col_ncm]
                mercosul_val = row[col_mercosul]
                
                if pd.notna(ncm_val) and pd.notna(mercosul_val):
                    all_rows.append({
                        'table_num': num,
                        'ncm_raw': str(ncm_val).strip(),
                        'mercosul_raw': str(mercosul_val).strip()
                    })

    # Guardar en archivo JSON de caché
    with open(cache_json_path, 'w', encoding='utf-8') as f:
        json.dump(all_rows, f, ensure_ascii=False, indent=2)

    print(f"✅ Extracción completada. Datos guardados en: {cache_json_path}")
    return all_rows

def load_rules_from_cache(pdf_path, cache_json_path):
    """Carga los datos desde la caché si existe, sino genera la caché primero."""
    if not os.path.exists(cache_json_path):
        return extract_and_build_cache(pdf_path, cache_json_path)
    
    with open(cache_json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [3]:
def extract_ncms_from_text(raw_text):
    """
    Busca patrones de NCM válidos (formatos XXXX.XX.XX o de 8 dígitos)
    en el texto de la celda antes de aplicar cualquier limpieza masiva.
    """
    text = str(raw_text).strip()
    # Captura patrones de NCM estándar (ej: 3006.10.90, 30061090)
    ncm_matches = re.findall(r'\b\d{4}\.\d{2}\.\d{2}\b|\b\d{8}\b', text)
    return [re.sub(r'[^\d]', '', m) for m in ncm_matches]

def expand_ncm_pattern(raw_text):
    """
    Analiza la celda NCM de forma estricta evitando falsos rangos provocados por descripciones.
    """
    text = str(raw_text).strip()
    
    # 1. Caso Rango explícito: "0401.10.10 a 0401.40.10" o "7208.10.00 a 7208.90.00"
    match_range = re.search(r'(\d{4}\.\d{2}\.\d{2}|\d{8})\s+a\s+(\d{4}\.\d{2}\.\d{2}|\d{8})', text, re.IGNORECASE)
    if match_range:
        start_digits = re.sub(r'[^\d]', '', match_range.group(1))
        end_digits = re.sub(r'[^\d]', '', match_range.group(2))
        return [{
            'type': 'RANGE',
            'start': int(start_digits),
            'end': int(end_digits),
            'raw': text
        }]
    
    # 2. Caso Capítulo: "Capítulo 30" o "Capítulo 1(*)" o "ex Capítulo 15"
    match_cap = re.search(r'Cap[íi]tulo\s*(\d{1,2})', text, re.IGNORECASE)
    if match_cap:
        cap_num = match_cap.group(1).zfill(2)
        return [{'type': 'CHAPTER', 'chapter': cap_num, 'raw': text}]

    # 3. Extraer todos los NCMs explícitos presentes en la celda (soporta "e", "ex", comas)
    explicit_ncms = extract_ncms_from_text(text)
    if explicit_ncms:
        return [{'type': 'EXACT', 'ncm': ncm_code, 'raw': text} for ncm_code in explicit_ncms]

    # 4. Caso Partida (4 dígitos) o Subpartida (6 dígitos) aisladas
    match_partida = re.search(r'\b\d{4}\.\d{2}\b|\b\d{4}\b', text)
    if match_partida:
        cleaned = re.sub(r'[^\d]', '', match_partida.group(0))
        start = int(cleaned.ljust(8, '0'))
        end = int(cleaned.ljust(8, '9'))
        return [{'type': 'RANGE', 'start': start, 'end': end, 'raw': text}]

    return [{'type': 'OTHER', 'raw': text}]


def parse_mercosul_rule(rule_str):
    """
    Extrae criterios arancelarios completos (MP, MS, MSP, CC), porcentaje MaxMNO y procesos.
    """
    rule_str = str(rule_str).strip()
    parsed = {
        "raw_rule": rule_str,
        "criteria": [],
        "max_mno_percentage": None,
        "chemical_processes": [],
        "operator": "SINGLE",
        "de_minimis_applies": True
    }
    
    # Extraer MaxMNO %
    mno_match = re.search(r'MaxMNO\s*(\d+)%', rule_str, re.IGNORECASE)
    if mno_match:
        parsed["max_mno_percentage"] = int(mno_match.group(1))

    # Captura precisa de siglas sin confundir abreviaturas compuestas como MSP
    if re.search(r'\bMSP\b', rule_str):
        parsed["criteria"].append("MSP")
    else:
        if re.search(r'\bMP\b', rule_str):
            parsed["criteria"].append("MP")
        if re.search(r'\bMS\b', rule_str):
            parsed["criteria"].append("MS")
    
    if re.search(r'\bCC\b', rule_str):
        parsed["criteria"].append("CC")

    # Procesos Químicos / Biotecnológicos (frecuentes en Secciones VI y VII)
    processes = [
        "Reação química", "Purificação", "Separação isomérica", 
        "Mudança de tamanho de partícula", "Produção de materiais padronizados", 
        "Processo biotecnológico"
    ]
    for proc in processes:
        if proc.lower() in rule_str.lower():
            parsed["chemical_processes"].append(proc)

    # Operadores
    if ' ou ' in rule_str.lower():
        parsed["operator"] = "OR"
    elif ' mais ' in rule_str.lower():
        parsed["operator"] = "AND"

    if 'não se aplica de minimis' in rule_str.lower():
        parsed["de_minimis_applies"] = False

    return parsed


def search_ncm_fast(search_ncm, rules_data):
    """
    Filtra y busca coincidencias priorizando la jerarquía arancelaria estricta.
    """
    target_clean = re.sub(r'[^\d]', '', str(search_ncm))
    if len(target_clean) != 8:
        raise ValueError("El NCM a buscar debe tener exactamente 8 dígitos.")

    target_int = int(target_clean)
    target_chapter = target_clean[:2]

    matches = []

    for row in rules_data:
        patterns = expand_ncm_pattern(row['ncm_raw'])
        
        for p in patterns:
            # Prioridad 1: Coincidencia Exacta de 8 dígitos
            if p['type'] == 'EXACT' and p['ncm'] == target_clean:
                matches.append({'priority': 1, 'row': row, 'pattern': p})
            
            # Prioridad 2: Rango Numérico Estricto
            elif p['type'] == 'RANGE' and p['start'] <= target_int <= p['end']:
                matches.append({'priority': 2, 'row': row, 'pattern': p})
            
            # Prioridad 3: Regla de Capítulo
            elif p['type'] == 'CHAPTER' and p['chapter'] == target_chapter:
                matches.append({'priority': 3, 'row': row, 'pattern': p})

    if not matches:
        return {
            "search_query": {"ncm": search_ncm, "chapter": target_chapter},
            "status": "NOT_FOUND"
        }

    # Seleccionar la coincidencia de mayor prioridad
    best_match = sorted(matches, key=lambda x: x['priority'])[0]
    matched_row = best_match['row']

    return {
        "search_query": {
            "ncm": search_ncm,
            "chapter": target_chapter
        },
        "match_info": {
            "match_type": best_match['pattern']['type'],
            "matched_ncm_expression": matched_row['ncm_raw'],
            "table_index": matched_row['table_num']
        },
        "mercosul_rule": parse_mercosul_rule(matched_row['mercosul_raw'])
    }

In [4]:
file_path = r"C:\Users\juan.s\Documents\Cerdificados_de_origem\dados\ACE_018_221_pt (1).pdf"
cache_path = r"C:\Users\juan.s\Documents\Cerdificados_de_origem\dados\reos_ace18_cache.json"

# Cargar las reglas (solo procesa el PDF con Docling si la caché no existe)
rules_db = load_rules_from_cache(file_path, cache_path)

# Búsqueda ultra rápida
res_1 = search_ncm_fast('3006.10.90', rules_db)
print(json.dumps(res_1, indent=2, ensure_ascii=False))

# Búsqueda subsecuente instantánea
res_2 = search_ncm_fast('8450.20.20', rules_db)
print(json.dumps(res_2, indent=2, ensure_ascii=False))

⚡ Procesando PDF con Docling (esto solo ocurrirá la primera vez)...


[INFO] 2026-09-15 14:52:43,815 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 14:52:43,843 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 14:52:43,849 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-09-15 14:52:46,362 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-09-15 14:52:46,766 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\juan.s\Documents\Cerdificados_de_origem\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-15 14:52:46,772 [RapidOCR] main.py:50: Using C:\Users\juan.s\Documents\Cerdificados_de_origem\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-15 14:52:47,214 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 14:52:47,215 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 14:52:47,217 [Rapi

2026-09-15 14:55:14,649 MatchingPostProcessor WARNING  Orphan pdf_cell 208 recovered to col=0 by nearest-column fallback (row=22, x=326.1, dist=121.5)
2026-09-15 14:55:14,656 MatchingPostProcessor WARNING  Orphan pdf_cell 215 recovered to col=0 by nearest-column fallback (row=23, x=326.1, dist=121.5)
2026-09-15 14:55:22,976 MatchingPostProcessor WARNING  Orphan pdf_cell 146 recovered to row=2 by nearest-row fallback (col=0, y=1071.0, dist=104.4)
2026-09-15 14:55:22,978 MatchingPostProcessor WARNING  Orphan pdf_cell 147 recovered to row=2 by nearest-row fallback (col=0, y=1071.0, dist=104.4)
2026-09-15 14:55:22,980 MatchingPostProcessor WARNING  Orphan pdf_cell 148 recovered to row=2 by nearest-row fallback (col=0, y=1071.0, dist=104.4)
2026-09-15 14:55:22,980 MatchingPostProcessor WARNING  Orphan pdf_cell 149 recovered to row=2 by nearest-row fallback (col=0, y=1071.0, dist=104.4)
2026-09-15 14:55:22,981 MatchingPostProcessor WARNING  Orphan pdf_cell 150 recovered to row=2 by nearest-r

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `do

✅ Extracción completada. Datos guardados en: C:\Users\juan.s\Documents\Cerdificados_de_origem\dados\reos_ace18_cache.json
{
  "search_query": {
    "ncm": "3006.10.90",
    "chapter": "30"
  },
  "match_info": {
    "match_type": "EXACT",
    "matched_ncm_expression": "3006.10.90 Aplica-se exclusivamente a: - Tecidos de malha de largura não superior a 30 cm, contendo em peso, 5% ou mais de fios elastômeros ou de fios de borracha, exceto: os veludos e pelúcias (incluídos os tecidos denominados de 'felpa longa' ou de 'pêlo comprido') e tecidos atoalhados (tecidos de anéis), de malha. - Aos tecidos de malha de largura não superior a 30 cm, exceto: os veludos e pelúcias (incluídos os tecidos denominados de 'felpa longa' ou de 'pêlo comprido') e tecidos atoalhados (tecidos de anéis), de malha. - Outros tecidos de malha- urdidura (incluídos os obtidos em teares para galões) de fibras artificiais, exceto: os veludos e pelúcias (incluídos os tecidos denominados de 'felpa longa' ou de 'pêlo com

In [5]:
#2827.60.12
res_2 = search_ncm_fast('2827.60.12', rules_db)
print(json.dumps(res_2, indent=2, ensure_ascii=False))

{
  "search_query": {
    "ncm": "2827.60.12",
    "chapter": "28"
  },
  "match_info": {
    "match_type": "CHAPTER",
    "matched_ncm_expression": "ex Capítulo 28",
    "table_index": 2
  },
  "mercosul_rule": {
    "raw_rule": "MSP ou MaxMNO 45% ou Reação química",
    "criteria": [
      "MSP"
    ],
    "max_mno_percentage": 45,
    "chemical_processes": [
      "Reação química"
    ],
    "operator": "OR",
    "de_minimis_applies": true
  }
}
